# 怎样用卷积网络改善图像识别？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

(1077, 1, 8, 8)


{
  "mlp_accuracy": 0.9583333333333334,
  "cnn_accuracy": 0.9555555555555556,
  "mlp_parameters": 4810,
  "cnn_parameters": 3818,
  "selected_by": "validation loss",
  "elapsed_seconds": 4.735
}


## 1. 补全：计算特征图形状
把一个全零批次逐层送进 CNN，断言第一卷积和池化之后的形状。

In [2]:
net = build_cnn()
z = torch.zeros(2, 1, 8, 8)
z = net[0](z)
assert z.shape == (2, 8, 8, 8)
z = net[2](net[1](z))
assert z.shape == (2, 8, 4, 4)
print(z.shape)

torch.Size([2, 8, 4, 4])


## 2. 修改：移到左边
写一个零填充左平移函数，用非对称小图核对，不得环绕边缘。

In [3]:
def shift_left(batch):
    out = torch.zeros_like(batch)
    out[:, :, :, :-1] = batch[:, :, :, 1:]
    return out
a = torch.arange(8.0).reshape(1, 1, 2, 4)
b = shift_left(a)
assert torch.equal(b[0, 0, 0], torch.tensor([1.0, 2.0, 3.0, 0.0]))
assert b[:, :, :, -1].sum() == 0
print(b)

tensor([[[[1., 2., 3., 0.],
          [5., 6., 7., 0.]]]])


## 3. 迁移：检查采集位置变化
在测试图像统一右移一格后评价同一个 CNN；不重新训练，比较原始和扰动准确率。

In [4]:
with torch.no_grad():
    shifted = cnn(shift_right(torch.tensor(X[te]))).argmax(1).numpy()
score = float(accuracy_score(y[te], shifted))
assert len(shifted) == len(te) and 0 <= score <= 1
print('原始', accuracy_score(y[te], p_cnn), '右移', score)

原始 0.9555555555555556 右移 0.9055555555555556
